# Phase 1 : Setup

In [57]:
import pandas as pd 
import warnings
warnings.filterwarnings('ignore')

## Load & Understand Data

In [146]:
df = pd.read_csv('netflix_titles.csv')

In [147]:
print("Shape", df.shape)
print("\nColumns: \n", df.columns.tolist())
print("\nData Types: \n", df.dtypes)
print("\n Missing Values \n", df.isnull().sum())
print(df.head(3))

Shape (8807, 12)

Columns: 
 ['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'description']

Data Types: 
 show_id           str
type              str
title             str
director          str
cast              str
country           str
date_added        str
release_year    int64
rating            str
duration          str
listed_in         str
description       str
dtype: object

 Missing Values 
 show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64
  show_id     type                 title         director  \
0      s1    Movie  Dick Johnson Is Dead  Kirsten Johnson   
1      s2  TV Show         Blood & Water              NaN   
2      s3  TV Show             Ganglands  Julien Leclercq   

         

# Phase 2: Data Cleaning

## Handle missing directors , cast & country

In [148]:
df['director'] = df['director'].fillna('Unknown')
df['cast'] = df['cast'].fillna("Unknown")
df['country'] = df['country'].fillna("Unknown")
df = df.dropna(subset=['duration'])

## Convert date_added to datetime

In [149]:
df['date_added']  = pd.to_datetime(df['date_added'].str.strip())

df['year_added'] = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month


## Fix the duration column

In [150]:
#Error: cannot convert float NaN to integer 
# so can use Int64
# but i rather dropped the missing value first

movies = df[df['type']== 'Movie'].copy()
shows = df[df['type'] == 'TV Show'].copy()

movies= movies.dropna(subset=['duration']).copy()
movies['duration_mins'] = movies['duration'].str.replace('min','').astype(int)

shows['duration_seasons'] = shows['duration'].str.replace('Season','').str.replace('s','').astype(int)

## Drop remaining nulls & duplicates

In [151]:
df.isnull().sum()

show_id          0
type             0
title            0
director         0
cast             0
country          0
date_added      10
release_year     0
rating           4
duration         0
listed_in        0
description      0
year_added      10
month_added     10
dtype: int64

In [152]:
df.dropna(subset=['rating'], inplace=True)

df.drop_duplicates(inplace=True)

print("Clean shape :", df.shape)
print("Remaining nulls: \n ", df.isnull().sum())

Clean shape : (8800, 14)
Remaining nulls: 
  show_id          0
type             0
title            0
director         0
cast             0
country          0
date_added      10
release_year     0
rating           0
duration         0
listed_in        0
description      0
year_added      10
month_added     10
dtype: int64


# Phase 3 : Core Analysis

## Movies vs TV Shows

In [153]:
content_counts = df['type'].value_counts()
print(content_counts)


total_movies = content_counts['Movie']
total_content = len(df)


movie_percentage = round((total_movies/total_content) * 100,1)


print("\nNetflix has ", movie_percentage, "% Movies")


print(f"\nNetflix has {content_counts['TV Show']/len(df)*100: .1f}% TV Shows")


type
Movie      6126
TV Show    2674
Name: count, dtype: int64

Netflix has  69.6 % Movies

Netflix has  30.4% TV Shows


## Top content-producing countries

In [154]:
# country = United States, Nepal
# explode() takes that list and creates a duplicate row for the movie—one for the US, and one for Nepal. This ensures both countries get proper credit in the final count.

top_countries = (df[df['country'] != 'Unknown'] ['country']
                   .str.split(', ' )
                   .explode()
                   .value_counts()
                   .head(10))

print(top_countries)

country
United States     3686
India             1046
United Kingdom     804
Canada             445
France             393
Japan              317
Spain              232
South Korea        231
Germany            226
Mexico             169
Name: count, dtype: int64


## Content growth over years

In [155]:
yearly = df.groupby(['year_added', 'type']).size().unstack(fill_value=0).astype(int)
print(yearly.tail(5))

type        Movie  TV Show
year_added                
2017.0        836      349
2018.0       1237      411
2019.0       1424      592
2020.0       1284      595
2021.0        993      505


## Most common ratings

In [156]:
ratings = df['rating'].value_counts()
print(ratings)

rating
TV-MA       3207
TV-14       2160
TV-PG        863
R            799
PG-13        490
TV-Y7        334
TV-Y         307
PG           287
TV-G         220
NR            80
G             41
TV-Y7-FV       6
NC-17          3
UR             3
Name: count, dtype: int64


## Average movie duration

In [157]:
avg_duration = movies['duration_mins'].mean()

print(f"\n Average movie duration is{avg_duration : .1f} mins")


season_avg = shows['duration_seasons'].mean()
print(f"\n Average TV shows duration(seasons) {season_avg: .1f}")


 Average movie duration is 99.6 mins

 Average TV shows duration(seasons)  1.8


## What % of content was added in each year?

In [158]:


yearly_per = (df['year_added'].value_counts(normalize=True)*100).round(3)


print(yearly_per.sort_index())


year_added
2008.0     0.023
2009.0     0.023
2010.0     0.011
2011.0     0.148
2012.0     0.034
2013.0     0.125
2014.0     0.273
2015.0     0.933
2016.0     4.846
2017.0    13.481
2018.0    18.749
2019.0    22.935
2020.0    21.377
2021.0    17.042
Name: proportion, dtype: float64


## What are the top 10 most common genres on Netflix

In [159]:
# listed_in =  International TV Shows, TV Dramas, TV Mysteries

genres = (df['listed_in']
          .str.split(', ')
          .explode()
          .value_counts()
          .head(10))


print(genres)

listed_in
International Movies        2752
Dramas                      2426
Comedies                    1674
International TV Shows      1350
Documentaries                869
Action & Adventure           859
TV Dramas                    763
Independent Movies           756
Children & Family Movies     641
Romantic Movies              616
Name: count, dtype: int64
